## Validación del funcionamiento de `client.py`:
1. **Configuración**: Parámetros de conexión desde `config.yaml`.
2. **Instanciación**: Creación del objeto `CenaceClient`.
3. **Petición HTTP**: Descarga de 1 día para 1 nodo (`06MON-115`) en mercado **MDA**.
4. **Inspección de JSON**: Verificación de la estructura del diccionario devuelto por CENACE.
5. **Aislamiento de nombres**: Verificación de que diferentes nodos no sobreescriben la caché.

In [10]:
import sys
from pathlib import Path
from datetime import date
import yaml

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.extract.client import CenaceClient

print("Directorio raíz:", ROOT_DIR)


Directorio raíz: /home/jose/prueba/challenge


## Cargar parámetros de conexión (`config.yaml`)

In [11]:
config_path = ROOT_DIR / "config.yaml"
with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("Sistema eléctrico:", config["system"])
print("Base URL:", config["api"]["base_url"])
print(f"Nodos configurados ({len(config['nodes'])}):", config["nodes"])


Sistema eléctrico: SIN
Base URL: https://ws01.cenace.gob.mx:8082/SWPML/SIM
Nodos configurados (9): ['06AEO-115', '06AHM-400', '06FRO-115', '06GRA-115', '06MON-115', '06MON-230', '06RGS-115', '06TKS-115', '06XOC-115']


In [12]:
raw_dir = ROOT_DIR / config["paths"]["raw_data_dir"]

client = CenaceClient(
    base_url=config["api"]["base_url"],
    system=config["system"],
    raw_data_dir=raw_dir,
    timeout_seconds=config["api"]["timeout_seconds"],
    max_retries=config["api"]["max_retries"],
    backoff_factor=config["api"]["backoff_factor"],
    rate_limit_delay_seconds=0.5,
)


## Petición a la API (1 nodo, 1 día)
Hacemos una petición real para el nodo `06MON-115` el día `2024-01-01`.

In [13]:
test_node = ["06MON-115"]
test_date = date(2024, 1, 1)

print(f"Consultando CENACE para nodo {test_node} en fecha {test_date}...")
response = client.fetch_window(
    process="MDA",
    nodes=test_node,
    start=test_date,
    end=test_date,
    force_reload=True,
)

print("\n--- Respuesta del Servidor ---")
print("Status:", response.get("status"))
print("Sistema:", response.get("sistema"))
print("Proceso:", response.get("proceso"))
print("Claves disponibles en el JSON:", list(response.keys()))


Consultando CENACE para nodo ['06MON-115'] en fecha 2024-01-01...

--- Respuesta del Servidor ---
Status: OK
Sistema: SIN
Proceso: MDA
Claves disponibles en el JSON: ['nombre', 'proceso', 'sistema', 'area', 'Resultados', 'status']


## Inspección de la estructura del JSON

In [14]:
resultados = response.get("Resultados", [])
print(f"Total de nodos devueltos: {len(resultados)}")

if resultados:
    nodo_info = resultados[0]
    print("Nodo reportado:", nodo_info.get("clv_nodo"))
    valores = nodo_info.get("Valores", [])
    print(f"Horas devueltas: {len(valores)}")
    print("\nMuestra de las primeras 3 horas:")
    for v in valores[:3]:
        print(" ", v)
else:
    print("Sin resultados en el payload.")


Total de nodos devueltos: 1
Nodo reportado: 06MON-115
Horas devueltas: 24

Muestra de las primeras 3 horas:
  {'fecha': '2024-01-01', 'hora': '1', 'pml': '279.54', 'pml_ene': '370.71', 'pml_per': '-44.07', 'pml_cng': '-47.09'}
  {'fecha': '2024-01-01', 'hora': '2', 'pml': '278.89', 'pml_ene': '311.75', 'pml_per': '-32.86', 'pml_cng': '0'}
  {'fecha': '2024-01-01', 'hora': '3', 'pml': '273.94', 'pml_ene': '305.52', 'pml_per': '-31.57', 'pml_cng': '0'}


## Aislamiento de caché para múltiples nodos
Comprobamos que al consultar más de un nodo se genera un nombre distinto para no sobreescribir datos.

In [15]:
cache_path = client._get_cache_path("MDA", test_node, test_date, test_date)
dos_nodos = ["06AEO-115", "06MON-115"]
ruta_multi = client._get_cache_path("MDA", dos_nodos, test_date, test_date)

print("Archivo 1 nodo:    ", cache_path.name)
print("Archivo multi-nodo:", ruta_multi.name)
assert cache_path.name != ruta_multi.name, "¡Error: las rutas colisionan!"
print("\nVerificación exitosa: las rutas de caché están aisladas.")


Archivo 1 nodo:     sin_mda_06MON-115_2024-01-01_2024-01-01.json
Archivo multi-nodo: sin_mda_2n_687aea39_2024-01-01_2024-01-01.json

Verificación exitosa: las rutas de caché están aisladas.
